# 91 Veeva Link business lists (Phase 3: B0, B1, B4)

Implements `90_veeva_link_phase3_working_plan.md` tasks B0, B1 and B4 (decisions P3-D1 to P3-D8). **Offline only** (P3-D1): files for brand, sales operations, Field Analytics and Medical Affairs, not rep-facing until X3 (Q1, Q2).

| Section | Output (`OUT/business/...`) | Checks |
| --- | --- | --- |
| B0 list base at C22 (`2026-10_offset7`), one row per NPI | used by B1 and `92_` | M1, M2 |
| B1 untargeted-authority list: Non-target x best decile 8-10 | `b1_untargeted_authority/run=<ts>/{list, summary}` | M3, M6, M8 |
| B4 low-adoption IDN experts: non-writing experts in IDNs with peer writer share <= 5% | `b4_low_adoption_idn_experts/run=<ts>/{list, by_idn}` | M4, M6, M8 |

Rules: expert = best in-scope decile 8 to 10 (snapshot T3); writer = KER-10 `active_nbrx_writers` > 0; segment = `cp_ker_hcp_2026q4` `NOTE_2` (FF CR1/CR2/CR4/CR5, Kerendia); specialty, name, city, state from BEST_AFFL (`BRAND_RANK = 1`). Payment columns are never selected (M6).

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np, datetime as dt

C = "`ph-com-ai-us-prod-catalog-2553213839599380`"
OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
P2, PB = OUT + "features_phase2/", OUT + "business/"
MV = f"{C}.kerendia_metric_views.mv_kerendia_nbrx"
COHORT = "2026-10_offset7"                       # list date = latest NBA 3.1 cohort (P3-D2)
VEEVA_CUTOFF = dt.date(2025, 3, 6)
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
INPUT_RUNS = {"t3_cross": "82_ run 20261123T093214", "t1_t2": "71_ run 20261123T085209", "ker10_test": "82_veeva_colocation_idn_signal_test_ker_10 run 20261123T095952",
              "segments": "cp_ker_hcp_2026q4"}

ker_last = spark.sql(f"SELECT MAX(`week_ending`) AS mx FROM (SELECT `week_ending`, MEASURE(`paid_nbrx`) AS n FROM {MV} WHERE `week_ending` >= '2026-09-01' GROUP BY ALL)").first().mx
W12_START = (pd.Timestamp(ker_last) - pd.Timedelta(weeks=52) + pd.Timedelta(days=1)).date()   # 12 months to the last KER-10 week
def ker10(a, b):
    return (spark.sql(f"""SELECT `provider_id` AS PROVIDER_ID, MEASURE(`active_nbrx_writers`) AS W FROM {MV}
                          WHERE `week_ending` >= '{a}' AND `week_ending` <= '{b}' GROUP BY ALL""").filter("W > 0").select("PROVIDER_ID"))

T3C = ["veeva_coverage_status", "veeva_best_decile", "veeva_decile_bin", "veeva_cl_decile_ckd", "veeva_cl_decile_hf", "veeva_cl_decile_t2d",
       "veeva_cl_decile_t1d", "veeva_cl_decile_htn", "veeva_cl_decile_mra_class", "veeva_cl_decile_sglt2_class", "veeva_fa_decile_max_inscope",
       "veeva_dr_any_inscope", "veeva_index_publication", "veeva_index_trial", "veeva_index_event", "veeva_index_guideline", "veeva_scientific_reach"]
t3 = spark.read.parquet(P2 + "t3").filter(F.col("COHORT_ID") == COHORT)
snap = t3.filter("version = 'snapshot'").select("NPI_NUMBER", "PROVIDER_ID", *T3C)
asof = t3.filter("version = 'asof'").select("NPI_NUMBER", F.col("veeva_best_decile").alias("_asof_best"))
xw = spark.table(f"{C}.kerendia_nba_3.source_provider_bhid_npi_mapping").select("PROVIDER_ID", "BH_ID").dropDuplicates(["PROVIDER_ID"])
affl = (spark.table(f"{C}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static").filter("BRAND_RANK = 1")
             .select("BH_ID", "HCP_FIRST_NAME", "HCP_LAST_NAME", "HCP_SPECIALTY", "CITY", "STATE", F.col("BHO_ID").alias("L1_ID"), "HCO_NAME")
             .dropDuplicates(["BH_ID"]))
hier = (spark.table(f"{C}.pace_ker_in_house_schema.mdm_stg_acct_hrchy_ker")
             .select(F.col("L1_CHILD_BHO_ID").alias("L1_ID"), "L4_IDN_BHO_ID", "L4_IDN_ORGNAME")
             .withColumn("L4_ID", F.when(F.col("L4_IDN_ORGNAME").isNull() | F.upper(F.trim("L4_IDN_ORGNAME")).isin("NA", "MULTIPLE DOCTORS"), None).otherwise(F.col("L4_IDN_BHO_ID")))
             .select("L1_ID", "L4_ID", F.col("L4_IDN_ORGNAME").alias("IDN_NAME")).dropDuplicates(["L1_ID"]))
seg = (spark.table(f"{C}.kerendia_cp.cp_ker_hcp_2026q4").filter(F.col("FF").isin("CR1", "CR2", "CR4", "CR5") &
                                     ((F.upper("PRODUCT_1") == "KERENDIA") | (F.upper("PRODUCT_2") == "KERENDIA")))
            .select(F.col("BAYER_ID").cast("string").alias("BH_ID"), F.col("NOTE_2").alias("SEGMENT")).dropDuplicates(["BH_ID"]))
lrx = spark.table(f"{C}.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7").filter("PERIOD_TYPE = 'W'")
mkt = (lrx.filter("upper(PRODUCT_GROUP) <> 'KERENDIA'").filter(F.col("PERIOD_ID").between(str(W12_START), str(ker_last)))
          .groupBy("PROVIDER_ID").agg(F.sum("TRX").alias("MKT_TRX_12M")))
spec_lrx = (lrx.filter(F.col("PERIOD_ID") >= str(W12_START)).groupBy("PROVIDER_ID")
               .agg(F.max_by("SPECIALTY_GROUP_ALIGNED", "PERIOD_ID").alias("_spec_lrx"), F.max_by("STATE", "PERIOD_ID").alias("_state_lrx")))

b = (snap.join(asof, "NPI_NUMBER", "left").join(xw, "PROVIDER_ID", "left").join(affl, "BH_ID", "left").join(hier, "L1_ID", "left")
         .join(seg, "BH_ID", "left").join(ker10(W12_START, ker_last).withColumn("KER10_WRITER_12M", F.lit(1)), "PROVIDER_ID", "left")
         .join(mkt, "PROVIDER_ID", "left").join(spec_lrx, "PROVIDER_ID", "left")
         .fillna(0, ["KER10_WRITER_12M", "MKT_TRX_12M"]).fillna("Non-target", ["SEGMENT"])
         .withColumn("SPECIALTY", F.coalesce("HCP_SPECIALTY", "_spec_lrx")).withColumn("STATE", F.coalesce("STATE", "_state_lrx"))
         .withColumn("veeva_score_from_stale_load", (F.col("veeva_best_decile").isNotNull() & ~F.col("veeva_best_decile").eqNullSafe(F.col("_asof_best"))).cast("int"))
         .drop("_asof_best", "_spec_lrx", "_state_lrx", "HCP_SPECIALTY"))
base = b.toPandas()
q = base.loc[base.MKT_TRX_12M > 0, "MKT_TRX_12M"].quantile([0.25, 0.5, 0.75]).values
base["MKT_VOL_BAND"] = np.where(base.MKT_TRX_12M <= 0, "0", pd.cut(base.MKT_TRX_12M, [0, *q, np.inf], labels=["Q1", "Q2", "Q3", "Q4"]).astype(str))
IND = {"CKD": "veeva_cl_decile_ckd", "HF": "veeva_cl_decile_hf", "T2D": "veeva_cl_decile_t2d", "T1D": "veeva_cl_decile_t1d", "HTN": "veeva_cl_decile_htn",
       "MRA class": "veeva_cl_decile_mra_class", "SGLT2 class": "veeva_cl_decile_sglt2_class"}
base["INDICATIONS"] = base[list(IND.values())].ge(8).apply(lambda r: ", ".join(k for k, c in IND.items() if r[c]), axis=1)
base["EXPERT"] = (base.veeva_best_decile.fillna(0) >= 8).astype(int)
base["TIER"] = np.select([base.veeva_best_decile >= 9, base.veeva_best_decile == 8], ["Top (9-10)", "High (8)"], "")
base["veeva_data_cutoff"] = str(VEEVA_CUTOFF)
base["veeva_data_age_months"] = round((pd.Timestamp(ker_last) - pd.Timestamp(VEEVA_CUTOFF)).days / 30.44, 1)
print(f"B0 rows {len(base):,} | NPIs {base.NPI_NUMBER.nunique():,} | KER-10 last week {ker_last} | 12M window {W12_START}..{ker_last}",
      f"| KER-10 writers 12M {int(base.KER10_WRITER_12M.sum()):,} | experts {int(base.EXPERT.sum()):,} | stale-load scores {int(base.veeva_score_from_stale_load.sum()):,}",
      f"| specialty filled {base.SPECIALTY.notna().mean():.1%} | data age {base.veeva_data_age_months.iloc[0]} months")
display(base.veeva_coverage_status.value_counts().rename("NPIs").to_frame())

In [0]:
B1_COLS = ["NPI_NUMBER", "PROVIDER_ID", "BH_ID", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE", "HCO_NAME", "IDN_NAME", "SEGMENT",
           "veeva_best_decile", "TIER", "INDICATIONS", "veeva_fa_decile_max_inscope", "KER10_WRITER_12M", "MKT_TRX_12M", "MKT_VOL_BAND",
           "veeva_score_from_stale_load", "veeva_data_cutoff", "veeva_data_age_months"]
b1 = base[(base.SEGMENT == "Non-target") & (base.EXPERT == 1)][B1_COLS].copy()
# priority inside the list: top tier, then non-writers, then market volume band (R2: decile adds beyond volume), then decile
b1["_vol"] = b1.MKT_VOL_BAND.map({"Q4": 4, "Q3": 3, "Q2": 2, "Q1": 1, "0": 0})
b1 = b1.sort_values(["TIER", "KER10_WRITER_12M", "_vol", "veeva_best_decile"], ascending=[False, True, False, False]).drop(columns="_vol")
b1["PRIORITY_RANK"] = np.arange(1, len(b1) + 1)
b1["WRITER_STATUS"] = np.where(b1.KER10_WRITER_12M == 1, "KER-10 writer (12M)", "Non-writer (12M)")

ex = b1.assign(IND=b1.INDICATIONS.replace("", "(none from CL)").str.split(", ")).explode("IND")   # one HCP can sit in several indications (P3-D4)
b1_sum = (ex.groupby(["SPECIALTY", "IND", "TIER", "WRITER_STATUS"], dropna=False).NPI_NUMBER.nunique().rename("HCPS").reset_index()
            .sort_values("HCPS", ascending=False))
print(f"B1 HCPs {len(b1):,} | top tier 9-10 {int((b1.TIER == 'Top (9-10)').sum()):,} | non-writers {int((b1.KER10_WRITER_12M == 0).sum()):,}",
      f"| stale-load scores {int(b1.veeva_score_from_stale_load.sum()):,}")
display(b1.groupby(["TIER", "WRITER_STATUS"]).size().rename("HCPS").reset_index())
print("By indication (HCP counted once per indication; do not sum):")
display(ex.groupby("IND").NPI_NUMBER.nunique().rename("HCPS").sort_values(ascending=False).reset_index())
print("Top 10 specialties:")
display(b1.SPECIALTY.fillna("(missing)").value_counts().head(10).rename("HCPS").reset_index())

In [0]:
# provider grain, same universe and rules as 82_veeva_colocation_idn_signal_test_ker_10 (self excluded)
UNIVERSE_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps"
univ = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
exp_p = snap.select("PROVIDER_ID", (F.coalesce("veeva_best_decile", F.lit(0)) >= 8).cast("int").alias("EXPERT")).dropDuplicates(["PROVIDER_ID"])
pb = (univ.join(xw, "PROVIDER_ID", "left").join(affl.select("BH_ID", "L1_ID"), "BH_ID", "left").join(hier, "L1_ID", "left")
          .join(seg, "BH_ID", "left").join(exp_p, "PROVIDER_ID", "left").fillna(0, ["EXPERT"]).fillna("Non-target", ["SEGMENT"]))

def low_adoption(lb_a, lb_b):
    d = pb.join(ker10(lb_a, lb_b).withColumn("LB_WRITER", F.lit(1)), "PROVIDER_ID", "left").fillna(0, ["LB_WRITER"]).toPandas()
    g, has = d.L4_ID, d.L4_ID.notna()
    d["IDN_PEERS"] = np.where(has, d.groupby("L4_ID").PROVIDER_ID.transform("size") - 1, np.nan)
    d["IDN_PEER_WRITERS"] = np.where(has, d.groupby("L4_ID").LB_WRITER.transform("sum") - d.LB_WRITER, np.nan)
    d["IDN_PEER_WRITER_SHARE"] = d.IDN_PEER_WRITERS / d.IDN_PEERS.replace(0, np.nan)
    low = d[(d.EXPERT == 1) & (d.LB_WRITER == 0) & (d.IDN_PEERS >= 1) & (d.IDN_PEER_WRITER_SHARE <= 0.05)].copy()
    return d, low

# M4 first: the KER-10 run windows (look-back 2025-07-25 to 2025-10-23)
_, low61 = low_adoption("2025-07-25", "2025-10-23")
m4 = {"experts": len(low61), "idns": low61.L4_ID.nunique(), "non_target": int((low61.SEGMENT == "Non-target").sum())}
print("M4 reproduce (expected 18,254 / 2,454 / 10,450):", m4)

# list date: same 13-week look-back, ending at the last KER-10 week
LB_B = ker_last; LB_A = (pd.Timestamp(ker_last) - pd.Timedelta(weeks=13) + pd.Timedelta(days=1)).date()
d22, low22 = low_adoption(LB_A, LB_B)
npi_cols = base[["PROVIDER_ID", "NPI_NUMBER", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE", "veeva_best_decile", "TIER", "INDICATIONS",
                 "KER10_WRITER_12M", "veeva_score_from_stale_load"]].drop_duplicates("PROVIDER_ID")
b4 = low22.merge(npi_cols, on="PROVIDER_ID", how="left")[["PROVIDER_ID", "NPI_NUMBER", "BH_ID", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE",
          "L4_ID", "IDN_NAME", "SEGMENT", "veeva_best_decile", "TIER", "INDICATIONS", "IDN_PEERS", "IDN_PEER_WRITERS", "IDN_PEER_WRITER_SHARE",
          "KER10_WRITER_12M", "veeva_score_from_stale_load"]]
b4["LOOKBACK"] = f"{LB_A}..{LB_B}"
b4["veeva_data_cutoff"] = str(VEEVA_CUTOFF)
idn = d22[d22.L4_ID.notna()].groupby(["L4_ID", "IDN_NAME"]).agg(IDN_HCPS=("PROVIDER_ID", "size"), IDN_WRITERS=("LB_WRITER", "sum"), IDN_EXPERTS=("EXPERT", "sum")).reset_index()
idn["IDN_WRITER_SHARE"] = idn.IDN_WRITERS / idn.IDN_HCPS
seg_mix = b4.pivot_table(index="L4_ID", columns="SEGMENT", values="PROVIDER_ID", aggfunc="count", fill_value=0).add_prefix("LIST_")
b4_idn = (b4.groupby("L4_ID").size().rename("LIST_EXPERTS").to_frame().join(seg_mix).reset_index().merge(idn, on="L4_ID", how="left")
            .sort_values("LIST_EXPERTS", ascending=False))
print(f"B4 at list date (look-back {LB_A}..{LB_B}): experts {len(b4):,} | IDNs {b4.L4_ID.nunique():,} | Non-target {int((b4.SEGMENT == 'Non-target').sum()):,}")
display(b4.SEGMENT.value_counts().rename("HCPS").reset_index())
display(b4_idn.head(10)[["IDN_NAME", "LIST_EXPERTS", "IDN_HCPS", "IDN_WRITERS", "IDN_WRITER_SHARE"]])
display(b4.SPECIALTY.fillna("(missing)").value_counts().head(8).rename("HCPS").reset_index())

In [0]:
def write(df, path):
    x = df.copy()
    for c in x.columns:
        x[c] = x[c].astype(str).replace({"nan": None, "None": None}) if x[c].dtype == object else x[c].astype(float)
    x["run_ts"] = RUN_TS
    spark.createDataFrame(x).write.mode("overwrite").parquet(path)

for name, df in {"b0_list_base": base, "b1_untargeted_authority/list": b1, "b1_untargeted_authority/summary": b1_sum,
                 "b4_low_adoption_idn_experts/list": b4, "b4_low_adoption_idn_experts/by_idn": b4_idn}.items():
    root, _, leaf = name.partition("/")
    write(df, f"{PB}{root}/run={RUN_TS}/{leaf or 'list'}")

# M8 independent recount on 100 random rows each, straight from the raw T3 parquet, segment table and metric view (Spark)
rs = b1.sample(100, random_state=1)[["NPI_NUMBER", "PROVIDER_ID", "veeva_best_decile", "KER10_WRITER_12M"]]
chk_t3 = (spark.read.parquet(P2 + f"t3/version=snapshot/COHORT_ID={COHORT}").filter(F.col("NPI_NUMBER").isin(rs.NPI_NUMBER.tolist()))
               .select("NPI_NUMBER", F.col("veeva_best_decile").alias("d2")).toPandas())
chk_w = set(r.PROVIDER_ID for r in spark.sql(f"""SELECT `provider_id` AS PROVIDER_ID, MEASURE(`active_nbrx_writers`) AS W FROM {MV}
          WHERE `week_ending` >= '{W12_START}' AND `week_ending` <= '{ker_last}' AND `provider_id` IN ({','.join(repr(p) for p in rs.PROVIDER_ID)}) GROUP BY ALL""").filter("W > 0").collect())
m = rs.merge(chk_t3, on="NPI_NUMBER", how="left")
m8_b1 = int((m.veeva_best_decile != m.d2).sum() + (rs.KER10_WRITER_12M != rs.PROVIDER_ID.isin(chk_w).astype(int)).sum())
r4 = b4.sample(100, random_state=1)
dw = d22.set_index("PROVIDER_ID")
m8_b4 = 0
for _, r in r4.iterrows():
    peers = d22[(d22.L4_ID == r.L4_ID) & (d22.PROVIDER_ID != r.PROVIDER_ID)]
    m8_b4 += int(len(peers) != r.IDN_PEERS or int(peers.LB_WRITER.sum()) != r.IDN_PEER_WRITERS or dw.loc[r.PROVIDER_ID, "LB_WRITER"] != 0 or dw.loc[r.PROVIDER_ID, "EXPERT"] != 1)

cov = base.veeva_coverage_status.value_counts()
export_cols = set(b1.columns) | set(b4.columns) | set(b1_sum.columns) | set(b4_idn.columns)
chk = pd.DataFrame([
    {"check": "M1", "name": "one row per NPI", "value": f"{len(base)} rows / {base.NPI_NUMBER.nunique()} NPIs", "pass": len(base) == base.NPI_NUMBER.nunique() == 294542},
    {"check": "M2", "name": "coverage = Phase 0", "value": cov.to_dict(), "pass": cov.get("SCORED") == 147487 and cov.get("IN_VEEVA_UNSCORED") == 81441 and cov.get("NOT_IN_VEEVA") == 65614},
    {"check": "M3", "name": "B1 within 1% of 18,208 / 10,113", "value": f"{len(b1)} / {int((b1.TIER == 'Top (9-10)').sum())}",
     "pass": abs(len(b1) / 18208 - 1) <= 0.01 and abs((b1.TIER == "Top (9-10)").sum() / 10113 - 1) <= 0.01},
    {"check": "M4", "name": "B4 reproduces KER-10 run 18,254 / 2,454 / 10,450", "value": m4, "pass": m4 == {"experts": 18254, "idns": 2454, "non_target": 10450}},
    {"check": "M6", "name": "no payment column in exports", "value": sorted(c for c in export_cols if "pay" in c.lower()), "pass": not any("pay" in c.lower() for c in export_cols)},
    {"check": "M8", "name": "100-row recount B1 + B4, mismatches", "value": f"B1 {m8_b1} | B4 {m8_b4}", "pass": m8_b1 == 0 and m8_b4 == 0}])
chk["value"] = chk.value.astype(str); chk["run_ts"] = RUN_TS; chk["inputs"] = str(INPUT_RUNS)
spark.createDataFrame(chk.astype(str)).write.mode("overwrite").parquet(f"{PB}checks/run={RUN_TS}/b0_b1_b4")
display(chk)
print("ALL PASS:", bool(chk["pass"].all()), "| run:", RUN_TS, "| written under", PB)